In [1]:
# @title 0) 저장소 클론·pip 설치 (로컬에서는 생략 가능)
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
WORKDIR = Path(os.environ.get("COLAB_REPO_DIR", "/content/colab"))
MARK = WORKDIR / "src" / "mindscopex_analysis" / "__init__.py"


def _run(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


if MARK.is_file():
    print(f"이미 클론됨 → git pull 만 실행: {WORKDIR}")
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=False)
else:
    WORKDIR.parent.mkdir(parents=True, exist_ok=True)
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    _run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)])

os.environ["MINDSCOPEX_ROOT"] = str(WORKDIR.resolve())
os.chdir(WORKDIR)
print("cwd =", os.getcwd())
print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"])

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
        "torch",
        "transformers",
        "accelerate",
        "pandas",
        "pyyaml",
        "tqdm",
    ]
)
print("pip install -e . 완료")


+ git clone --depth 1 https://github.com/JeonDongJun/mindscopex_analysis /content/colab
cwd = /content/colab
MINDSCOPEX_ROOT = /content/colab
pip install -e . 완료


# Qwen 직관·논리·CRT 추론 및 결과 저장

`configs/notebook_defaults.yaml`과 **아래 설정 셀에서 고른 실험 YAML**을 병합합니다 (`models`, `model_tags`, `problems` 등은 전부 YAML). 기본 파일은 **Qwen2.5-7B-Instruct vs DeepSeek-R1-Distill-Qwen-7B** 비교용입니다. 다른 실험은 `YAML_PATH`만 바꾸면 됩니다.


## 설정 로드


In [2]:
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# 프로젝트
_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
        raise FileNotFoundError(f"MINDSCOPEX_ROOT={env!r} 아래에 {_REPO_MARK.as_posix()} 가 없습니다.")
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError(
        "src/mindscopex_analysis 을 찾지 못했습니다. Colab에서는 clone 후 프로젝트 루트로 이동하세요."
    )


_SRC = _find_repo_root() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from mindscopex_analysis.notebook_utils import (
    dtype_from_str,
    load_notebook_experiment_config,
    project_root_from_notebook,
)

print("준비 완료.")


준비 완료.


In [ ]:
ROOT = project_root_from_notebook(Path.cwd())
# 소형 모델 실험: "configs/qwen_reasoning_tasks.yaml"
YAML_PATH = ROOT / "configs" / "qwen7_deepseek_r1_distill_compare.yaml"

EXP = load_notebook_experiment_config(ROOT, YAML_PATH)
if not EXP.get("models"):
    raise ValueError("실험 YAML에 models 가 필요합니다.")

GEN = EXP.get("generation") or {}
OUT_CFG = EXP.get("output") or {}
MODELS = EXP["models"]
RUNTIME = EXP["runtime"]

DEVICE = RUNTIME.get("device") or ("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = dtype_from_str(str(RUNTIME.get("dtype", "float32")))
TRC = bool(RUNTIME.get("trust_remote_code", True))

MODEL_KEYS = list(MODELS.keys())
_mlist = EXP.get("model_tags")
_one = EXP.get("model_tag")
if _mlist:
    MODEL_TAGS_TO_RUN = list(_mlist)
elif _one:
    MODEL_TAGS_TO_RUN = [_one]
else:
    MODEL_TAGS_TO_RUN = [MODEL_KEYS[0]]

for t in MODEL_TAGS_TO_RUN:
    if t not in MODELS:
        raise ValueError(f"model_tags 에 {t!r} 가 없습니다. 가능: {MODEL_KEYS}")

tasks: list[dict] = []
problems = EXP.get("problems")
if problems:
    for p in problems:
        pid = p.get("id") or f"{p.get('category', 'x')}_{len(tasks)}"
        tasks.append(
            {
                "id": pid,
                "category": p.get("category", "general"),
                "prompt": p["prompt"],
                "verification": p.get("verification") or {},
            }
        )
else:
    pgroups = EXP.get("prompt_groups") or {}
    for cat, plist in pgroups.items():
        for i, text in enumerate(plist):
            tasks.append(
                {
                    "id": f"{cat}_{i}",
                    "category": cat,
                    "prompt": text,
                    "verification": {},
                }
            )

if not tasks:
    raise ValueError("실행할 문제가 없습니다. YAML에 problems 또는 prompt_groups 를 채우세요.")

SUBDIR = str(OUT_CFG.get("subdir", "qwen_reasoning_runs"))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
OUT_DIR = ROOT / "outputs" / SUBDIR / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXP_NAME = EXP.get("experiment_name") or YAML_PATH.stem
print(f"실험={EXP_NAME!r}  파일={YAML_PATH.name}  DEVICE={DEVICE}")
print("model_tags:", MODEL_TAGS_TO_RUN)
for t in MODEL_TAGS_TO_RUN:
    print(f"  - {t}: {MODELS[t]!r}")
print(f"문제 수: {len(tasks)}  →  결과 디렉터리: {OUT_DIR}")


preset='qwen_reasoning_pair'  DEVICE=cuda
model_tags: ['non_reasoning', 'reasoning']
  - non_reasoning: 'Qwen/Qwen2.5-0.5B-Instruct'
  - reasoning: 'Qwen/Qwen3-0.6B'
문제 수: 11  →  결과 디렉터리: /content/colab/outputs/qwen_reasoning_runs/20260419_112723_utc


## 모델 순차 로드 (VRAM)

다음 셀에서 `model_tags` 순서로 모델을 **하나씩** 로드했다가 문제 전부 생성 후 해제합니다. 두 모델을 동시에 올리지 않습니다.


In [4]:
max_length = int(RUNTIME.get("max_length", 2048))
print("토크나이저·모델은 다음 '추론·검증·저장' 셀에서 model_tags 마다 순차 로드됩니다.")


토크나이저·모델은 다음 '추론·검증·저장' 셀에서 model_tags 마다 순차 로드됩니다.


## 추론·검증·저장


In [5]:
def _verify(text: str, spec: dict) -> tuple[str, str]:
    """(상태, 메시지) — pass / fail / skip"""
    if not spec:
        return "skip", "검증 규칙 없음"
    t = text.lower()
    any_kw = spec.get("expect_contains_any") or []
    if any_kw and not any(k.lower() in t for k in any_kw):
        return "fail", f"다음 중 하나도 없음: {any_kw}"
    all_kw = spec.get("expect_contains_all") or []
    if all_kw and not all(k.lower() in t for k in all_kw):
        return "fail", f"다음을 모두 포함하지 않음: {all_kw}"
    rx = spec.get("expect_regex")
    if rx:
        try:
            if not re.search(rx, text, re.DOTALL):
                return "fail", f"정규식 불일치: {rx!r}"
        except re.error as e:
            return "fail", f"정규식 오류: {e}"
    return "pass", "규칙 충족"


def _format_input(tok, system_prompt: str, user_prompt: str) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    if hasattr(tok, "apply_chat_template"):
        return tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    if system_prompt:
        return f"{system_prompt}\n\nUser: {user_prompt}\nAssistant:"
    return f"User: {user_prompt}\nAssistant:"


import gc

system_prompt = str(GEN.get("system_prompt") or "")
max_new_tokens = int(GEN.get("max_new_tokens", 512))
temperature = float(GEN.get("temperature", 0.7))
top_p = float(GEN.get("top_p", 0.9))
do_sample = bool(GEN.get("do_sample", True))
extra_gen = GEN.get("extra_generate_kwargs") or {}

rows: list[dict] = []

for tag in MODEL_TAGS_TO_RUN:
    mid = MODELS[tag]
    print(f"\n{'=' * 60}\n▶ 로드: {tag}  →  {mid}\n{'=' * 60}", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(mid, trust_remote_code=TRC)
    model = AutoModelForCausalLM.from_pretrained(
        mid,
        torch_dtype=DTYPE,
        trust_remote_code=TRC,
    ).to(DEVICE).eval()
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    for task in tqdm(tasks, desc=f"{tag}"):
        prompt_text = task["prompt"]
        fmt = _format_input(tokenizer, system_prompt, prompt_text)
        inputs = tokenizer(fmt, return_tensors="pt", truncation=True, max_length=max_length)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if do_sample:
            gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"] = top_p
        gen_kwargs.update(extra_gen)

        with torch.no_grad():
            out_ids = model.generate(**inputs, **gen_kwargs)

        new_tokens = out_ids[0, inputs["input_ids"].shape[1] :]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        status, vmsg = _verify(answer, task.get("verification") or {})

        rows.append(
            {
                "id": task["id"],
                "category": task["category"],
                "prompt": prompt_text,
                "answer": answer,
                "verify_status": status,
                "verify_detail": vmsg,
                "model_tag": tag,
                "model_id": mid,
            }
        )

    del model, tokenizer
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
try:
    from IPython.display import display
except ImportError:
    display = print
display(df)

snap_path = OUT_DIR / "config_snapshot.yaml"
with snap_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(EXP, f, allow_unicode=True, sort_keys=False)

df.to_csv(OUT_DIR / "results.csv", index=False, encoding="utf-8-sig")
df.to_json(OUT_DIR / "results.json", orient="records", force_ascii=False, indent=2)

model_summary = ", ".join(f"`{MODELS[t]}` ({t})" for t in MODEL_TAGS_TO_RUN)
md_lines = [
    f"# Qwen 추론 실행 ({RUN_ID})",
    "",
    f"- 실험: `{EXP_NAME}`",
    f"- 설정 파일: `{YAML_PATH.name}`",
    f"- 모델: {model_summary}",
    "",
    "| id | model_tag | category | verify | prompt (요약) |",
    "|----|-----------|----------|--------|---------------|",
]
for _, r in df.iterrows():
    pshort = (r["prompt"][:50] + "…") if len(str(r["prompt"])) > 50 else r["prompt"]
    md_lines.append(
        f"| {r['id']} | {r['model_tag']} | {r['category']} | {r['verify_status']} | {pshort} |"
    )

md_lines.append("")
md_lines.append("## 상세")
md_lines.append("")
for _, r in df.iterrows():
    md_lines.append(f"### {r['model_tag']} — {r['id']}")
    md_lines.append("")
    md_lines.append(f"- 검증: **{r['verify_status']}** — {r['verify_detail']}")
    md_lines.append("")
    md_lines.append("**질문**")
    md_lines.append("")
    md_lines.append(str(r["prompt"]))
    md_lines.append("")
    md_lines.append("**생성**")
    md_lines.append("")
    md_lines.append(str(r["answer"]))
    md_lines.append("")

(OUT_DIR / "README.md").write_text("\n".join(md_lines), encoding="utf-8")
print(f"저장 완료: {OUT_DIR}")



▶ 로드: non_reasoning  →  Qwen/Qwen2.5-0.5B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

non_reasoning:   0%|          | 0/11 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 결과 파일 다운로드 (엑셀)

**구글 드라이브 마운트는 필수가 아닙니다.** 런타임 VM에 `results.csv`가 이미 저장되어 있고, UTF-8 BOM이라 **엑셀에서 바로 열 수 있습니다.**

- Colab 왼쪽 **파일** 아이콘 → `outputs/.../results.csv` 우클릭 → 다운로드  
- 또는 아래 셀을 실행하면 브라우저 저장 창이 뜹니다.  
- 세션이 끊기면 VM이 지워지므로, **영구 보관**할 때만 드라이브에 복사하면 됩니다.

In [ ]:
# Colab: 브라우저로 results.csv 받기 (드라이브 마운트 불필요)
try:
    from google.colab import files

    csv_path = OUT_DIR / "results.csv"
    print("다운로드:", csv_path.name)
    files.download(str(csv_path))
except ImportError:
    print("로컬 Jupyter — 파일 경로:", OUT_DIR / "results.csv")

# 선택: .xlsx (pandas + openpyxl; 없으면 CSV만 쓰면 됨)
try:
    xlsx_path = OUT_DIR / "results.xlsx"
    df.to_excel(xlsx_path, index=False, engine="openpyxl")
    print("저장:", xlsx_path)
    try:
        from google.colab import files

        files.download(str(xlsx_path))
    except ImportError:
        pass
except ImportError:
    print("(선택) .xlsx 는 pip install openpyxl 후 재실행 — 또는 results.csv 를 엑셀로 열기")

다운로드: results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

저장: /content/colab/outputs/qwen_reasoning_runs/20260419_084117_utc/results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>